# nnUNet Pipeline — Local GPU Runner

Dataset: **Dataset777_GCEF** | Samples: **mishmar_hanegev_Cu011_samp_2_Rec_nlm**, **nlm_volume** | Trainer: **nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss** | Config: **3d_fullres**

Prerequisites (run locally before this notebook):
1. `preprocessing_nnUNet_train.py` → produces `Dataset777_GCEF/` in `nnUNet_raw`
2. (For inference) `preprocessing_nnUNet_predict_tif.py` + `preprocessing_nnUNet_predict_split.py` → produces split chunks

Paths are resolved from `analysis/data_registry.json`. Do **not** edit Section 3 paths manually — change the registry instead.

Training samples loaded from registry: all samples with `status == "annotations_ready"`.


## 1) Runtime Setup

In [1]:
# Verify GPU runtime
!nvidia-smi

Thu Jun 18 09:55:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.61                 Driver Version: 572.61         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:3B:00.0 Off |                    0 |
| 30%   34C    P8             15W /  300W |     401MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

# Local repository directory
REPO_DIR = r'C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT'
assert os.path.isdir(REPO_DIR), f'Repo not found: {REPO_DIR}'
print('Repo dir:', os.listdir(REPO_DIR))


Repo dir: ['.claude', '.git', '.github', '.gitignore', '.vscode', 'analysis', 'chunk_extractor.py', 'colab_nnUNet_pipeline.ipynb', 'dataset_info.json', 'debug_labels_777.py', 'debug_labels_777_followup.py', 'extract_trainlog.py', 'Figures', 'Fiji_macros', 'inspect_predictions.py', 'legacy', 'LICENSE', 'litreture', 'make_annotations.py', 'make_psd_plot.py', 'merge_annotations.py', 'mishmar_psd.log', 'nnUNetTrainer_betterIgnoreSampling.py', 'otsu_threshold_3d.py', 'postprocessing_nnUNet_predict.py', 'postprocessing_nnUNet_predict_concatenate.py', 'postprocessing_pipeline.ipynb', 'preprocess', 'preprocessing_nnUNet_predict.py', 'preprocessing_nnUNet_predict_split.py', 'preprocessing_nnUNet_predict_tif.py', 'preprocessing_nnUNet_train.py', 'preprocess_playground', 'README.md', 'retrieve_dice_score.py', 'run_remaining_fullctx_overnight.ipynb', 'select_slices_and_predict.py', 'setup_prompt.md', 'training_diag', 'Utilities', '__path__.py', '__pycache__']


## 2) Register Custom Trainer

In [3]:
import shutil
import sys
import importlib
import nnunetv2
import os

# Find nnunetv2 trainers directory
nnunet_trainers_dir = os.path.join(
    os.path.dirname(nnunetv2.__file__),
    'training', 'nnUNetTrainer', 'variants', 'sampling'
)
os.makedirs(nnunet_trainers_dir, exist_ok=True)

src = os.path.join(REPO_DIR, 'nnUNetTrainer_betterIgnoreSampling.py')
dst = os.path.join(nnunet_trainers_dir, 'nnUNetTrainer_betterIgnoreSampling.py')
shutil.copy2(src, dst)
print(f'Copied: {src} → {dst}')

# Force reload the module to pick up the updated file
if 'nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling' in sys.modules:
    del sys.modules['nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling']

# Import and verify the new trainer
from nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling import (
    nnUNetTrainer_betterIgnoreSampling,
    nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss
)
print('✓ Custom trainer registered:', nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss.__name__)
print('✓ Base trainer registered:', nnUNetTrainer_betterIgnoreSampling.__name__)

Copied: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\nnUNetTrainer_betterIgnoreSampling.py → c:\Users\rony.schwartz\.conda\envs\venv-napari\Lib\site-packages\nnunetv2\training\nnUNetTrainer\variants\sampling\nnUNetTrainer_betterIgnoreSampling.py
✓ Custom trainer registered: nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss
✓ Base trainer registered: nnUNetTrainer_betterIgnoreSampling


## 3) Set Environment Variables & Paths

In [4]:
import os, json

# ── Data Registry ──────────────────────────────────────────────────────────────
REGISTRY_PATH = os.path.join(REPO_DIR, 'analysis', 'data_registry.json')
with open(REGISTRY_PATH) as _f:
    _registry = json.load(_f)

# All samples with annotations ready for training — these will be staged and trained together
TRAINING_SAMPLES = [
    s for s in _registry['samples']
    if s.get('status') in ('annotations_ready', 'ready_for_training')
]
assert TRAINING_SAMPLES, 'No samples with status in [annotations_ready, ready_for_training] found in registry'
print(f'Training samples ({len(TRAINING_SAMPLES)}):')
for s in TRAINING_SAMPLES:
    print(f"  {s['sample_id']}")
    print(f"    raw_tiff:   {s['raw_tiff_path']}")
    print(f"    annotation: {_registry['annotations']['active_latest'].get(s['sample_id'], 'MISSING')}")
    print(f"    status:     {s.get('status', 'UNKNOWN')}")

# ── Primary sample for single-volume ops (inference, etc.) ────────────────────
# Override SAMPLE_ID here if you want inference on a specific sample.
SAMPLE_ID = TRAINING_SAMPLES[0]['sample_id']
_sample_rec  = TRAINING_SAMPLES[0]

# Canonical paths from registry (all on HIVE)
RAW_TIFF_PATH    = _sample_rec['raw_tiff_path']
ANNOTATION_PATH  = _registry['annotations']['active_latest'][SAMPLE_ID]
RAW_TIFF_DIR     = os.path.dirname(RAW_TIFF_PATH)
ANNOTATION_DIR   = os.path.dirname(ANNOTATION_PATH)

print(f'\nPrimary sample (for inference): {SAMPLE_ID}')

# ── nnUNet workspace (HIVE) — shared across samples ───────────────────────────
HIVE_BASE   = r'\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources'
LOCAL_BASE  = os.path.join(HIVE_BASE, 'multi_sample_iter02')

TRAINER_NAME = 'nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss'

nnUNet_raw          = os.path.join(LOCAL_BASE, 'nnUNet_raw')
nnUNet_preprocessed = os.path.join(LOCAL_BASE, 'nnUNet_preprocessed')
nnUNet_results      = os.path.join(LOCAL_BASE, 'nnUNet_results')

os.environ['nnUNet_raw']          = nnUNet_raw
os.environ['nnUNet_preprocessed'] = nnUNet_preprocessed
os.environ['nnUNet_results']      = nnUNet_results
os.environ['nnUNet_compile']      = 'false'   # Triton not available on Windows

for d in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(d, exist_ok=True)

print()
print('LOCAL_BASE:',          LOCAL_BASE)
print('nnUNet_raw:',          nnUNet_raw)
print('nnUNet_preprocessed:', nnUNet_preprocessed)
print('nnUNet_results:',      nnUNet_results)
print('nnUNet_compile:',      os.environ['nnUNet_compile'])
print('TRAINER_NAME:',        TRAINER_NAME)

Training samples (2):
  mishmar_hanegev_Cu011_samp_2_Rec_nlm
    raw_tiff:   \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
    annotation: \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\bootstrap_outputs_fullctx\mishmar_hanegev_Cu011_samp_2_Rec_nlm\annotations_v20260610_r01\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
    status:     ready_for_training
  nlm_volume
    raw_tiff:   \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\nlm_volume.tif
    annotation: \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\annotations_v20260617_r01\nlm_volume.tif
    status:     ready_for_training

Primary sample (for inference): mishmar_hanegev_Cu011_samp_2_Rec_nlm

LOCAL_BASE: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02
nnUNet_raw: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_raw
nnUNet_preprocessed: \\hive3065\Yael_Mishael\Rony\rem

## 4) Verify Training Data

This notebook stages and preprocesses all `annotations_ready` samples loaded from the registry.

Expected output under HIVE (`LOCAL_BASE = multi_sample_iter01`):
```
<LOCAL_BASE>\nnUNet_raw\Dataset777_GCEF\
  imagesTr\mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz
  imagesTr\nlm_volume_0000.nii.gz
  labelsTr\mishmar_hanegev_Cu011_samp_2_Rec_nlm.nii.gz
  labelsTr\nlm_volume.nii.gz
  dataset.json
```


In [3]:
import subprocess, sys, os, glob, shutil

# Force a clean rebuild before training so latest annotations are guaranteed to be used.
# Set to False only if you intentionally want to reuse prepared artifacts.
FORCE_REBUILD = True

# Stage ALL training samples into a shared staging directory
stage_root   = os.path.join(LOCAL_BASE, '_multi_sample_stage')
stage_images = os.path.join(stage_root, 'images')
stage_masks  = os.path.join(stage_root, 'annotations')
os.makedirs(stage_images, exist_ok=True)
os.makedirs(stage_masks,  exist_ok=True)

# Build sets of expected filenames so we can detect stale files from removed samples
expected_image_names = set()
expected_mask_names  = set()

for s in TRAINING_SAMPLES:
    sid       = s['sample_id']
    raw_tif   = s['raw_tiff_path']
    ann_tif   = _registry['annotations']['active_latest'][sid]

    expected_image_names.add(os.path.basename(raw_tif))
    expected_mask_names.add(os.path.basename(ann_tif))

    dst_img = os.path.join(stage_images, os.path.basename(raw_tif))
    dst_msk = os.path.join(stage_masks,  os.path.basename(ann_tif))

    shutil.copy2(raw_tif, dst_img)
    shutil.copy2(ann_tif, dst_msk)
    print(f'Staged [{sid}]')
    print(f'  image: {dst_img}')
    print(f'  mask:  {dst_msk}')

# Remove any stale files left over from previous runs with different samples
for p in glob.glob(os.path.join(stage_images, '*.tif')) + glob.glob(os.path.join(stage_images, '*.tiff')):
    if os.path.basename(p) not in expected_image_names:
        os.remove(p)
        print(f'Removed stale image: {p}')
for p in glob.glob(os.path.join(stage_masks, '*.tif')) + glob.glob(os.path.join(stage_masks, '*.tiff')):
    if os.path.basename(p) not in expected_mask_names:
        os.remove(p)
        print(f'Removed stale mask: {p}')

# Patch __path__.py to point to the multi-sample staging dirs
_path_py = os.path.join(REPO_DIR, '__path__.py')
_path_py_content = (
    f'PATH_ImageJ = ""\n'
    f'PATH_nnUNet_raw = r"{nnUNet_raw}"\n'
    f'input_dir_images = r"{stage_images}"\n'
    f'input_dir_masks = r"{stage_masks}"\n'
)
with open(_path_py, 'w') as _f:
    _f.write(_path_py_content)
print('\nPatched __path__.py:')
print(_path_py_content)

dataset_raw_dir = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
preprocessed_dataset_dir = os.path.join(nnUNet_preprocessed, 'Dataset777_GCEF')

if FORCE_REBUILD:
    if os.path.isdir(dataset_raw_dir):
        shutil.rmtree(dataset_raw_dir)
        print(f'Removed previous raw dataset: {dataset_raw_dir}')
    if os.path.isdir(preprocessed_dataset_dir):
        shutil.rmtree(preprocessed_dataset_dir)
        print(f'Removed previous preprocessed dataset: {preprocessed_dataset_dir}')
    print('FORCE_REBUILD=True -> clean rebuild is enabled.')

print('Running preprocessing_nnUNet_train.py for multi-sample staged dirs...')
result = subprocess.run(
    [sys.executable, os.path.join(REPO_DIR, 'preprocessing_nnUNet_train.py')],
    cwd=REPO_DIR,
    env=os.environ,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    raise RuntimeError(f'preprocessing_nnUNet_train.py failed (exit {result.returncode})')
print('Preprocessing done.')


Staged [mishmar_hanegev_Cu011_samp_2_Rec_nlm]
  image: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\_multi_sample_stage\images\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
  mask:  \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\_multi_sample_stage\annotations\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
Staged [nlm_volume]
  image: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\_multi_sample_stage\images\nlm_volume.tif
  mask:  \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\_multi_sample_stage\annotations\nlm_volume.tif

Patched __path__.py:
PATH_ImageJ = ""
PATH_nnUNet_raw = r"\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\nnUNet_raw"
input_dir_images = r"\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\_multi_sample_stage\images"
input_dir_mask

In [4]:
# Verify all training samples are present in the dataset
dataset_dir = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
assert os.path.isdir(dataset_dir), f'Dataset folder not found: {dataset_dir}'

import glob, os
images = sorted(glob.glob(os.path.join(dataset_dir, 'imagesTr', '*_0000.nii.gz')))
labels = sorted(glob.glob(os.path.join(dataset_dir, 'labelsTr', '*.nii.gz')))
dataset_json = os.path.join(dataset_dir, 'dataset.json')

print(f'imagesTr: {len(images)} files')
print(f'labelsTr: {len(labels)} files')
print(f'dataset.json exists: {os.path.isfile(dataset_json)}')
for p in images:
    print('  image:', os.path.basename(p))
for p in labels:
    print('  label:', os.path.basename(p))

expected_imgs = sorted(f'{s["sample_id"]}_0000.nii.gz' for s in TRAINING_SAMPLES)
expected_lbls = sorted(f'{s["sample_id"]}.nii.gz'       for s in TRAINING_SAMPLES)

assert [os.path.basename(p) for p in images] == expected_imgs, f'imagesTr mismatch: {[os.path.basename(p) for p in images]} != {expected_imgs}'
assert [os.path.basename(p) for p in labels] == expected_lbls, f'labelsTr mismatch: {[os.path.basename(p) for p in labels]} != {expected_lbls}'
assert os.path.isfile(dataset_json), 'dataset.json missing'
print(f'✓ Dataset contains all {len(TRAINING_SAMPLES)} training samples.')


imagesTr: 2 files
labelsTr: 2 files
dataset.json exists: True
  image: mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz
  image: nlm_volume_0000.nii.gz
  label: mishmar_hanegev_Cu011_samp_2_Rec_nlm.nii.gz
  label: nlm_volume.nii.gz
✓ Dataset contains all 2 training samples.


## 5) nnUNet Planning & Preprocessing

In [5]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-m', 'nnunetv2.experiment_planning.plan_and_preprocess_entrypoints',
     '-d', '777', '--verify_dataset_integrity'],
    env=os.environ,
    timeout=3600,  # 1 hour timeout
    capture_output=False  # Don't stream, just let it run
)

if result.returncode != 0:
    raise RuntimeError(f"plan_and_preprocess failed (exit {result.returncode})")
print("✓ Planning and preprocessing complete.")


✓ Planning and preprocessing complete.


## 6) Training

Run one fold at a time. Change `FOLD` to train multiple folds sequentially (0–4).

**Warning**: each fold may take many hours on a single Colab GPU.

In [6]:
# PolyLRScheduler compatibility fix is applied directly to polylr.py on disk.
# No runtime patch needed here.
print("PolyLRScheduler: using patched polylr.py (PyTorch 2.x compatible)")


PolyLRScheduler: using patched polylr.py (PyTorch 2.x compatible)


In [7]:
# Create custom splits_final.json — all training samples in both train and val
import json, os

preprocessed_dir = os.path.join(os.environ['nnUNet_preprocessed'], 'Dataset777_GCEF')
sample_ids = [s['sample_id'] for s in TRAINING_SAMPLES]
splits = [{"train": sample_ids, "val": sample_ids}]

splits_path = os.path.join(preprocessed_dir, 'splits_final.json')
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"Wrote {splits_path}")
print(json.dumps(splits, indent=2))


Wrote \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter01\nnUNet_preprocessed\Dataset777_GCEF\splits_final.json
[
  {
    "train": [
      "mishmar_hanegev_Cu011_samp_2_Rec_nlm",
      "nlm_volume"
    ],
    "val": [
      "mishmar_hanegev_Cu011_samp_2_Rec_nlm",
      "nlm_volume"
    ]
  }
]


In [8]:
import subprocess, sys, os

# Early stopping controls (env-driven in trainer)
os.environ['NNUNET_EARLY_STOP_ENABLED'] = '1'
os.environ['NNUNET_EARLY_STOP_PATIENCE'] = '20'
os.environ['NNUNET_EARLY_STOP_MIN_DELTA'] = '0.001'
print('Early stopping env:', {
    'NNUNET_EARLY_STOP_ENABLED': os.environ['NNUNET_EARLY_STOP_ENABLED'],
    'NNUNET_EARLY_STOP_PATIENCE': os.environ['NNUNET_EARLY_STOP_PATIENCE'],
    'NNUNET_EARLY_STOP_MIN_DELTA': os.environ['NNUNET_EARLY_STOP_MIN_DELTA'],
})

# Stream training output live to the notebook cell
proc = subprocess.Popen(
    [sys.executable, '-m', 'nnunetv2.run.run_training',
     '777', '3d_fullres', '0', '-tr', TRAINER_NAME],
    env=os.environ,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"run_training failed (exit {proc.returncode})")


Early stopping env: {'NNUNET_EARLY_STOP_ENABLED': '1', 'NNUNET_EARLY_STOP_PATIENCE': '20', 'NNUNET_EARLY_STOP_MIN_DELTA': '0.001'}

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0
c:\Users\rony.schwartz\.conda\envs\venv-napari\Lib\site-packages\nnunetv2\training\nnUNetTrainer\nnUNetTrainer.py:161: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler() if self.device.type == 'cuda' else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for d

In [9]:
# List training results
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')

dataset.json
dataset_fingerprint.json
fold_0
plans.json


## 7) Training Outputs

Results are saved on HIVE at:
```
<LOCAL_BASE>\nnUNet_results\Dataset777_GCEF\<TRAINER_NAME>__nnUNetPlans__3d_fullres\
```
Use `extract_trainlog.py` to parse training logs and plot loss curves.


In [5]:
# Optional: list training result files
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')


analytics
dataset.json
dataset_fingerprint.json
fold_0
plans.json


In [6]:
# Training analytics: mean Dice, val loss, and train loss
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
 )

assert os.path.isdir(results_dir), f'Results dir not found: {results_dir}'

# Collect all training logs from fold folders
log_files = sorted(
    p for p in glob.glob(os.path.join(results_dir, 'fold_*', '*.txt'))
    if 'training_log' in os.path.basename(p).lower()
 )

if not log_files:
    raise FileNotFoundError(f'No training_log*.txt files found under {results_dir}')

print('Found log files:')
for lf in log_files:
    print(f'  {lf}')

epoch_re = re.compile(r'Epoch\s+(\d+)')
train_re = re.compile(r'train_loss\s+(-?[\d.eE]+)')
val_re = re.compile(r'val_loss\s+(-?[\d.eE]+)')
dice_re = re.compile(
    r'(?:EMA\s+pseudo\s+Dice|pseudo\s+Dice|mean\s+foreground\s+Dice)[^\d-]*(-?[\d.eE]+)',
    re.IGNORECASE,
 )

all_rows = []
for lf in log_files:
    fold_name = os.path.basename(os.path.dirname(lf))
    epoch_data = {}
    current_epoch = None

    with open(lf, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m_epoch = epoch_re.search(line)
            if m_epoch:
                current_epoch = int(m_epoch.group(1))
                if current_epoch not in epoch_data:
                    epoch_data[current_epoch] = {'epoch': current_epoch, 'fold': fold_name}

            if current_epoch is None:
                continue

            m_train = train_re.search(line)
            if m_train:
                epoch_data[current_epoch]['train_loss'] = float(m_train.group(1))

            m_val = val_re.search(line)
            if m_val:
                epoch_data[current_epoch]['val_loss'] = float(m_val.group(1))

            m_dice = dice_re.search(line)
            if m_dice:
                epoch_data[current_epoch]['mean_dice'] = float(m_dice.group(1))

    all_rows.extend(epoch_data.values())

df = pd.DataFrame(all_rows).sort_values(['epoch', 'fold'])
if df.empty:
    raise RuntimeError('Parsed zero epochs from training logs.')

summary = df.groupby('epoch', as_index=False).agg({
    'train_loss': 'mean',
    'val_loss': 'mean',
    'mean_dice': 'mean',
})

# Save summary for reproducibility
analytics_dir = os.path.join(results_dir, 'analytics')
os.makedirs(analytics_dir, exist_ok=True)
summary_csv = os.path.join(analytics_dir, 'training_metrics_summary.csv')
summary.to_csv(summary_csv, index=False)
print(f'\nSaved summary CSV: {summary_csv}')

# Plot metrics
fig, ax1 = plt.subplots(figsize=(10, 5))
x = summary['epoch']

# Loss curves on left axis
ax1.plot(x, summary['train_loss'], label='Train Loss', color='tab:blue', linewidth=2)
ax1.plot(x, summary['val_loss'], label='Val Loss', color='tab:orange', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.25)

# Mean Dice on right axis
ax2 = ax1.twinx()
if summary['mean_dice'].notna().any():
    ax2.plot(x, summary['mean_dice'], label='Mean Dice', color='tab:green', linewidth=2)
    ax2.set_ylabel('Mean Dice')
    ax2.set_ylim(0, 1)
else:
    print('Warning: mean_dice was not found in the logs; only loss curves are shown.')

# Combined legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

plt.title('Training Metrics by Epoch (mean across folds/logs)')
plt.tight_layout()
plt.show()

# Print latest metrics row
last = summary.iloc[-1]
print('\nLatest epoch metrics:')
print(f"  epoch={int(last['epoch'])}")
print(f"  train_loss={last['train_loss']:.6f}")
print(f"  val_loss={last['val_loss']:.6f}")
if pd.notna(last['mean_dice']):
    print(f"  mean_dice={last['mean_dice']:.6f}")

Found log files:
  \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\fold_0\training_log_2026_6_17_16_59_35.txt

Saved summary CSV: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\analytics\training_metrics_summary.csv

Latest epoch metrics:
  epoch=134
  train_loss=-0.545000
  val_loss=-0.591600
  mean_dice=0.627200


C:\Users\rony.schwartz\AppData\Local\Temp\58\ipykernel_59740\2715892243.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Save training plots to files
import os

analytics_dir = os.path.join(results_dir, 'analytics')

# Main combined plot (Loss + Dice)
fig, ax1 = plt.subplots(figsize=(12, 6))
x = summary['epoch']

ax1.plot(x, summary['train_loss'], label='Train Loss', color='tab:blue', linewidth=2.5)
ax1.plot(x, summary['val_loss'], label='Val Loss', color='tab:orange', linewidth=2.5)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
if summary['mean_dice'].notna().any():
    ax2.plot(x, summary['mean_dice'], label='Mean Dice', color='tab:green', linewidth=2.5)
    ax2.set_ylabel('Mean Dice', fontsize=12)
    ax2.set_ylim(0, 1)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best', fontsize=11)
plt.title('Training Metrics by Epoch', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save combined plot
combined_png = os.path.join(analytics_dir, 'training_metrics_combined.png')
combined_pdf = os.path.join(analytics_dir, 'training_metrics_combined.pdf')
plt.savefig(combined_png, dpi=150, bbox_inches='tight')
plt.savefig(combined_pdf, bbox_inches='tight')
plt.close()

# Separate plots
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, summary['train_loss'], label='Train Loss', color='tab:blue', linewidth=2, marker='o', markersize=4)
ax.plot(x, summary['val_loss'], label='Val Loss', color='tab:orange', linewidth=2, marker='s', markersize=4)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.title('Loss Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
loss_png = os.path.join(analytics_dir, 'training_loss_curves.png')
plt.savefig(loss_png, dpi=150, bbox_inches='tight')
plt.close()

if summary['mean_dice'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, summary['mean_dice'], label='Mean Dice', color='tab:green', linewidth=2.5, marker='D', markersize=5)
    ax.fill_between(x, summary['mean_dice'], alpha=0.2, color='tab:green')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Mean Dice', fontsize=12)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.title('Mean Dice Score', fontsize=13, fontweight='bold')
    plt.tight_layout()
    dice_png = os.path.join(analytics_dir, 'training_mean_dice.png')
    plt.savefig(dice_png, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'\nSaved Dice plot: {dice_png}')

print('\n✓ Training plots saved:')
print(f'  Combined: {combined_png}')
print(f'  Combined: {combined_pdf}')
print(f'  Loss:     {loss_png}')
print(f'\nAll files in analytics directory: {analytics_dir}')


Saved Dice plot: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\analytics\training_mean_dice.png

✓ Training plots saved:
  Combined: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\analytics\training_metrics_combined.png
  Combined: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\analytics\training_metrics_combined.pdf
  Loss:     \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\analytics\training_l

## 8) Inference Data

Inference is run per-sample. Set `INFERENCE_SAMPLE_ID` below to select which volume to predict.
Available samples: all entries in `TRAINING_SAMPLES`.

Split chunks are written to:
`<HIVE_BASE>\<INFERENCE_SAMPLE_ID>\inference_input\`


In [8]:
import subprocess, sys, os, glob
import numpy as np
import tifffile
import nibabel as nib
from tqdm import tqdm

# ── Select inference sample ────────────────────────────────────────────────────
# Change INFERENCE_SAMPLE_ID to run inference on a different sample.
INFERENCE_SAMPLE_ID = TRAINING_SAMPLES[0]['sample_id']
_inf_rec = next(s for s in TRAINING_SAMPLES if s['sample_id'] == INFERENCE_SAMPLE_ID)
INF_RAW_TIFF_PATH = _inf_rec['raw_tiff_path']

# Per-sample inference workspace (under HIVE, NOT under multi_sample LOCAL_BASE)
INF_SAMPLE_BASE = os.path.join(HIVE_BASE, INFERENCE_SAMPLE_ID)
NIFTI_DIR       = os.path.join(INF_SAMPLE_BASE, 'nifti_predict')
INFERENCE_INPUT = os.path.join(INF_SAMPLE_BASE, 'inference_input')
os.makedirs(NIFTI_DIR,       exist_ok=True)
os.makedirs(INFERENCE_INPUT, exist_ok=True)

# Model folder for plans.json (from the multi-sample training workspace)
MODEL_DIR = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)

print(f'Inference sample:  {INFERENCE_SAMPLE_ID}')
print(f'Raw TIF:           {INF_RAW_TIFF_PATH}')
print(f'NIfTI dir:         {NIFTI_DIR}')
print(f'Inference input:   {INFERENCE_INPUT}')
print(f'Model dir:         {MODEL_DIR}')

# --- Step 1: .tif -> _0000.nii.gz (direct Python, zscore norm) ---
print('\n=== Step 1: .tif -> _0000.nii.gz ===')
vol = tifffile.imread(INF_RAW_TIFF_PATH).astype(np.float32)
mean, std = vol.mean(), vol.std()
vol = (vol - mean) / (std + 1e-8)
vol = vol.transpose(2, 1, 0)  # (Z, Y, X) -> (X, Y, Z) for nibabel
stem = os.path.splitext(os.path.basename(INF_RAW_TIFF_PATH))[0]
out_path = os.path.join(NIFTI_DIR, f'{stem}_0000.nii.gz')
nib.save(nib.Nifti1Image(vol, affine=np.eye(4)), out_path)
print(f'  Saved: {out_path}  shape={vol.shape}')
print('Step 1 done.')

# --- Step 2: _0000.nii.gz -> split chunks -> INFERENCE_INPUT ---
print('=== Step 2: preprocessing_nnUNet_predict_split.py ===')
result = subprocess.run(
    [sys.executable,
     os.path.join(REPO_DIR, 'preprocessing_nnUNet_predict_split.py'),
     '-i', NIFTI_DIR,
     '-o', INFERENCE_INPUT,
     '-m', MODEL_DIR],
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    raise RuntimeError(f'preprocessing_nnUNet_predict_split.py failed (exit {result.returncode})')
print('Step 2 done.')


Inference sample:  mishmar_hanegev_Cu011_samp_2_Rec_nlm
Raw TIF:           \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
NIfTI dir:         \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nifti_predict
Inference input:   \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\inference_input
Model dir:         \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres

=== Step 1: .tif -> _0000.nii.gz ===
  Saved: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nifti_predict\mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz  shape=(650, 650, 822)
Step 1 done.
=== Step 2: preprocessing_nnUNet_predict_split.py ===
Model Parameters:
Patch S

In [9]:
INFERENCE_OUTPUT = os.path.join(INF_SAMPLE_BASE, 'inference_output')
os.makedirs(INFERENCE_OUTPUT, exist_ok=True)

assert os.path.isdir(INFERENCE_INPUT), f'Inference input not found: {INFERENCE_INPUT}'
input_files = [f for f in os.listdir(INFERENCE_INPUT) if f.endswith('_0000.nii.gz')]
print(f'Sample:                  {INFERENCE_SAMPLE_ID}')
print(f'Inference input dir:     {INFERENCE_INPUT}')
print(f'Inference output dir:    {INFERENCE_OUTPUT}')
print(f'Inference chunks found:  {len(input_files)}')


Sample:                  mishmar_hanegev_Cu011_samp_2_Rec_nlm
Inference input dir:     \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\inference_input
Inference output dir:    \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\inference_output
Inference chunks found:  8


## 9) Inference

In [10]:
import os

# Diagnose: what datasets exist in nnUNet_results?
print('=== nnUNet_results contents ===')
if os.path.isdir(nnUNet_results):
    datasets = os.listdir(nnUNet_results)
    if datasets:
        for d in sorted(datasets):
            print(f'  {d}')
            sub = os.path.join(nnUNet_results, d)
            for item in sorted(os.listdir(sub)):
                print(f'    {item}')
    else:
        print('  (empty — no training results found)')
else:
    print(f'  nnUNet_results dir does not exist: {nnUNet_results}')

# Check specifically for the expected fold checkpoint
expected = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres',
    'fold_0', 'checkpoint_final.pth'
)
print(f'\nExpected checkpoint exists: {os.path.isfile(expected)}')
print(f'Expected checkpoint path:   {expected}')


=== nnUNet_results contents ===
  Dataset777_GCEF
    nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres

Expected checkpoint exists: True
Expected checkpoint path:   \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\fold_0\checkpoint_final.pth


In [11]:
import os
import torch
import numpy
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

# nnUNet checkpoints were saved with PyTorch <2.6, which used weights_only=False.
# PyTorch 2.6 changed the default to True, breaking checkpoint loading.
# Patch torch.load to restore weights_only=False (safe: this is our own trained model).
_orig_torch_load = torch.load
def _patched_load(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(f, *args, **kwargs)
torch.load = _patched_load

MODEL_DIR = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
print(f'Model dir: {MODEL_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')

# Auto-select checkpoint: prefer final, fall back to best, then latest
fold_dir = os.path.join(MODEL_DIR, 'fold_0')
for candidate in ('checkpoint_final.pth', 'checkpoint_best.pth', 'checkpoint_latest.pth'):
    if os.path.isfile(os.path.join(fold_dir, candidate)):
        checkpoint_name = candidate
        break
else:
    raise FileNotFoundError(f'No checkpoint found in {fold_dir}')
print(f'Using checkpoint: {checkpoint_name}')

# More stable inference settings for notebooks/Windows to avoid kernel crashes.
# Keep CUDA for compute when available, but do preprocessing/postprocessing off-device.
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
perform_everything_on_device = False

if use_cuda:
    torch.cuda.empty_cache()

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=False,
    perform_everything_on_device=perform_everything_on_device,
    device=device,
    verbose=True,
    allow_tqdm=True
)

predictor.initialize_from_trained_model_folder(
    MODEL_DIR,
    use_folds=(0,),
    checkpoint_name=checkpoint_name
)

predictor.predict_from_files(
    INFERENCE_INPUT,
    INFERENCE_OUTPUT,
    save_probabilities=False,
    overwrite=True,
    num_processes_preprocessing=1,
    num_processes_segmentation_export=1,
    folder_with_segs_from_prev_stage=None,
    num_parts=1,
    part_id=0
)
print('Inference complete.')


Model dir: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres
CUDA available: True
Using checkpoint: checkpoint_final.pth
There are 8 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 8 cases that I would like to predict

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__0__167_:
perform_everything_on_device: False
Input shape: torch.Size([1, 167, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 200, image size is torch.Size([167, 650, 650]), tile_size [128, 128, 128], tile_step_size 0.5
steps:
[[0, 39], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 200/200 [00:11<00:00, 17.26it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__0__167_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__142__373_:
perform_everything_on_device: False
Input shape: torch.Size([1, 231, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 300, image size is torch.Size([231, 650, 650]), tile_size [128, 128, 128], tile_step_size 0.5
steps:
[[0, 52, 103], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 300/300 [00:16<00:00, 18.63it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__142__373_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__245__476_:
perform_everything_on_device: False
Input shape: torch.Size([1, 231, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 300, image size is torch.Size([231, 650, 650]), tile_size [128, 128, 128], tile_step_size 0.5
steps:
[[0, 52, 103], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


...

Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__451__682_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__554__785_:
perform_everything_on_device: False
Input shape: torch.Size([1, 231, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 300, image size is torch.Size([231, 650, 650]), tile_size [128, 128, 128], tile_step_size 0.5
steps:
[[0, 52, 103], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 300/300 [00:16<00:00, 18.28it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__554__785_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__657__822_:
perform_everything_on_device: False
Input shape: torch.Size([1, 165, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 200, image size is torch.Size([165, 650, 650]), tile_size [128, 128, 128], tile_step_size 0.5
steps:
[[0, 37], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 200/200 [00:10<00:00, 18.67it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__657__822_
Inference complete.


## 10) Validate Predictions


In [ ]:
# List prediction outputs
pred_files = [f for f in os.listdir(INFERENCE_OUTPUT) if f.endswith('.nii.gz')]
print(f'Predictions: {len(pred_files)} files')
for f in sorted(pred_files):
    print(f'  {f}')

Predictions: 8 files
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__0__167_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__142__373_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__245__476_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__348__579_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__39__270_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__451__682_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__554__785_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__657__822_.nii.gz


: 

In [ ]:
# Fresh inference + concatenation for ONE sample at a time (to avoid OOM crashes)
import glob
import json
import os
import subprocess
import sys
import gc

import nibabel as nib
import numpy as np
import tifffile
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

registry_path = os.path.join(REPO_DIR, 'analysis', 'data_registry.json')
with open(registry_path, 'r', encoding='utf-8') as f:
    registry = json.load(f)

resolved = registry.get('checks', {}).get('last_gate', {}).get('resolved_paths', {})
model_dir = resolved.get(
    'model_dir',
    os.path.join(
        nnUNet_results, 'Dataset777_GCEF',
        f'{TRAINER_NAME}__nnUNetPlans__3d_fullres',
    ),
)
assert os.path.isdir(model_dir), f'Model dir not found: {model_dir}'

print(f'Model dir: {model_dir}')
print(f'CUDA available: {torch.cuda.is_available()}')

# Process ONE sample at a time to avoid memory leaks
sample_ids = ['mishmar_hanegev_Cu011_samp_2_Rec_nlm', 'nlm_volume']
fresh_concat_paths = {}

for idx, sid in enumerate(sample_ids):
    print(f'\n{"="*100}')
    print(f'[SAMPLE {idx+1}/{len(sample_ids)}] {sid}')
    print(f'{"="*100}')
    
    # Clear GPU memory before each sample
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()
    
    sample_rec = next(s for s in registry['samples'] if s['sample_id'] == sid)
    raw_tif = sample_rec['raw_tiff_path']
    nifti_input_path = sample_rec['nifti_input_path']
    nifti_dir = os.path.dirname(nifti_input_path)
    split_dir = sample_rec['split_input_dir']
    pred_out_dir = sample_rec['prediction_output_dir']

    concat_dir_key = 'mishmar_new_concat_dir' if sid.startswith('mishmar_') else 'nlm_new_concat_dir'
    concat_dir = resolved.get(concat_dir_key)
    if not concat_dir:
        old_concat = sample_rec.get('prediction_concatenated_path')
        if not old_concat:
            raise ValueError(f'prediction_concatenated_path missing for {sid}')
        concat_dir = os.path.join(
            os.path.dirname(os.path.dirname(old_concat)),
            os.path.basename(os.path.dirname(old_concat)) + '_iter02',
        )
    os.makedirs(concat_dir, exist_ok=True)

    print(f'raw_tif:   {raw_tif}')
    print(f'nifti_dir: {nifti_dir}')
    print(f'split_dir: {split_dir}')
    print(f'pred_out:  {pred_out_dir}')
    print(f'concat:    {concat_dir}')

    os.makedirs(nifti_dir, exist_ok=True)
    os.makedirs(split_dir, exist_ok=True)
    os.makedirs(pred_out_dir, exist_ok=True)

    # Clean stale files in working dirs to guarantee fresh run artifacts
    for work_dir in (nifti_dir, split_dir, pred_out_dir):
        for p in glob.glob(os.path.join(work_dir, '*.nii.gz')):
            os.remove(p)

    # Step 1: TIFF -> NIfTI _0000
    print('\nStep 1: Loading TIFF and converting to NIfTI...')
    vol = tifffile.imread(raw_tif).astype(np.float32)
    mean, std = vol.mean(), vol.std()
    vol = (vol - mean) / (std + 1e-8)
    vol = vol.transpose(2, 1, 0)
    stem = os.path.splitext(os.path.basename(raw_tif))[0]
    out_nifti = os.path.join(nifti_dir, f'{stem}_0000.nii.gz')
    nib.save(nib.Nifti1Image(vol, affine=np.eye(4)), out_nifti)
    print(f'  ✓ Saved NIfTI: {out_nifti} shape={vol.shape}')
    del vol  # Free memory immediately

    # Step 2: split
    print('Step 2: Splitting into chunks...')
    split_cmd = [
        sys.executable,
        os.path.join(REPO_DIR, 'preprocessing_nnUNet_predict_split.py'),
        '-i', nifti_dir,
        '-o', split_dir,
        '-m', model_dir,
    ]
    res_split = subprocess.run(split_cmd, env=os.environ, capture_output=True, text=True)
    if res_split.returncode != 0:
        print(res_split.stdout)
        print(res_split.stderr)
        raise RuntimeError(f'Split failed for {sid} (exit {res_split.returncode})')
    print(f'  ✓ {res_split.stdout.count("Split")} chunks created')

    # Step 3: predict on split chunks (fresh predictor each time)
    print('Step 3: Running inference on chunks...')
    
    # PyTorch 2.6 checkpoint compatibility patch
    _orig_torch_load = torch.load
    def _patched_load(f, *args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(f, *args, **kwargs)
    torch.load = _patched_load

    fold_dir = os.path.join(model_dir, 'fold_0')
    for candidate in ('checkpoint_final.pth', 'checkpoint_best.pth', 'checkpoint_latest.pth'):
        candidate_path = os.path.join(fold_dir, candidate)
        if os.path.isfile(candidate_path):
            checkpoint_name = candidate
            break
    else:
        raise FileNotFoundError(f'No checkpoint found in {fold_dir}')
    
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=False,
        perform_everything_on_device=False,
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        verbose=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        model_dir, use_folds=(0,), checkpoint_name=checkpoint_name
    )

    predictor.predict_from_files(
        split_dir,
        pred_out_dir,
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
    print(f'  ✓ Inference complete')
    
    # Clean up predictor
    del predictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Step 4: concatenate split predictions
    print('Step 4: Concatenating predictions...')
    concat_cmd = [
        sys.executable,
        os.path.join(REPO_DIR, 'postprocessing_nnUNet_predict_concatenate.py'),
        '-i', pred_out_dir,
        '-o', concat_dir,
    ]
    res_concat = subprocess.run(concat_cmd, env=os.environ, capture_output=True, text=True)
    if res_concat.returncode != 0:
        print(res_concat.stdout)
        print(res_concat.stderr)
        raise RuntimeError(f'Concatenation failed for {sid} (exit {res_concat.returncode})')
    print(f'  ✓ Concatenation complete')

    concat_path = os.path.join(concat_dir, f'{sid}.nii.gz')
    if not os.path.isfile(concat_path):
        raise FileNotFoundError(f'Expected concatenated prediction missing: {concat_path}')
    fresh_concat_paths[sid] = concat_path
    print(f'  ✓ Fresh concatenated prediction saved: {concat_path}')

print('\n' + '='*100)
print('DONE. Fresh concatenated predictions:')
for sid, p in fresh_concat_paths.items():
    print(f'  {sid}:')
    print(f'    {p}')
print('='*100)

Model dir: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\multi_sample_iter02\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres
CUDA available: True

[SAMPLE 1/2] mishmar_hanegev_Cu011_samp_2_Rec_nlm
raw_tif:   \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
nifti_dir: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nifti_predict
split_dir: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\inference_input
pred_out:  \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\inference_output
concat:    \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\inference_output_concat_iter02

Step 1: Loading TIFF and converting to NIfTI...
  ✓ Saved NIfTI: